# 🔬 MedGraph Challenge: Baseline Training Tutorial

This notebook demonstrates how to:
1. Load and explore the cell-graph dataset
2. Train baseline GNN models (GCN, GAT, GraphSAGE)
3. Evaluate performance with official metrics
4. Create a valid submission file

---

## 1. Setup & Installation

In [ ]:
# Install dependencies (run once)
# !pip install torch torch-geometric torch-scatter torch-sparse
# !pip install numpy pandas scikit-learn matplotlib seaborn tqdm

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Add project root to path
sys.path.append('..')

from utils.dataset import MedGraphDataset
from baselines import GCNClassifier, GATClassifier, GraphSAGEClassifier
from evaluation.metrics import compute_metrics, print_metrics

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Load and Explore the Dataset

In [ ]:
# Load datasets
train_dataset = MedGraphDataset(root='../data', split='train')
val_dataset = MedGraphDataset(root='../data', split='val')
test_dataset = MedGraphDataset(root='../data', split='test')

print(f"Train: {len(train_dataset)} graphs")
print(f"Val:   {len(val_dataset)} graphs")
print(f"Test:  {len(test_dataset)} graphs")

In [ ]:
# Explore dataset statistics
stats = train_dataset.statistics()

print("\n📊 Dataset Statistics")
print("="*40)
print(f"Number of features: {stats['num_features']}")
print(f"Number of classes: {stats['num_classes']}")
print(f"\nClass Distribution:")
for name, count in stats['class_distribution'].items():
    print(f"  {name}: {count}")

print(f"\nGraph Sizes:")
print(f"  Nodes: {stats['nodes']['mean']:.1f} ± {stats['nodes']['std']:.1f}")
print(f"  Edges: {stats['edges']['mean']:.1f} ± {stats['edges']['std']:.1f}")

In [ ]:
# Visualize a sample graph
sample = train_dataset[0]

print(f"\n📈 Sample Graph")
print(f"  Nodes: {sample.num_nodes}")
print(f"  Edges: {sample.num_edges}")
print(f"  Features shape: {sample.x.shape}")
print(f"  Label: {sample.y.item()} ({['Normal', 'Benign', 'Malignant'][sample.y.item()]})")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
class_names = ['Normal', 'Benign', 'Malignant']
colors = ['#3fb950', '#ffd700', '#f85149']

counts = [stats['class_distribution'][name] for name in class_names]
axes[0].bar(class_names, counts, color=colors)
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution (Training Set)')

# Node distribution
node_counts = [data.num_nodes for data in train_dataset]
axes[1].hist(node_counts, bins=30, color='#58a6ff', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Number of Nodes')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Graph Size Distribution')

plt.tight_layout()
plt.show()

## 3. Create Data Loaders

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

## 4. Define Training Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=-1)
        correct += (pred == batch.y).sum().item()
        total += batch.num_graphs
    
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        
        total_loss += loss.item() * batch.num_graphs
        all_preds.extend(out.argmax(dim=-1).cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
    
    metrics = compute_metrics(np.array(all_labels), np.array(all_preds))
    metrics['loss'] = total_loss / len(loader.dataset)
    
    return metrics, np.array(all_preds)

## 5. Train a GCN Baseline

In [ ]:
# Hyperparameters
HIDDEN_CHANNELS = 256
NUM_LAYERS = 4
DROPOUT = 0.5
LEARNING_RATE = 0.001
EPOCHS = 50

# Initialize model
model = GCNClassifier(
    in_channels=train_dataset.num_node_features,
    hidden_channels=HIDDEN_CHANNELS,
    num_classes=train_dataset.num_classes,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

In [ ]:
# Training loop
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_f1': []}
best_val_f1 = 0

pbar = tqdm(range(1, EPOCHS + 1), desc='Training')

for epoch in pbar:
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    
    # Evaluate
    val_metrics, _ = evaluate(model, val_loader, criterion, device)
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_metrics['loss'])
    history['val_f1'].append(val_metrics['macro_f1'])
    
    # Save best model
    if val_metrics['macro_f1'] > best_val_f1:
        best_val_f1 = val_metrics['macro_f1']
        torch.save(model.state_dict(), 'best_model.pt')
    
    pbar.set_postfix({
        'Train Loss': f'{train_loss:.4f}',
        'Val F1': f'{val_metrics["macro_f1"]:.4f}',
        'Best F1': f'{best_val_f1:.4f}'
    })

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()

axes[1].plot(history['train_acc'], label='Train Accuracy')
axes[1].plot(history['val_f1'], label='Val Macro F1')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Accuracy / F1 Curves')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Final Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pt'))

# Evaluate on validation set
val_metrics, val_preds = evaluate(model, val_loader, criterion, device)
print_metrics(val_metrics)

In [ ]:
# Confusion matrix visualization
from sklearn.metrics import confusion_matrix

val_labels = [data.y.item() for data in val_dataset]
cm = confusion_matrix(val_labels, val_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=['Normal', 'Benign', 'Malignant'],
    yticklabels=['Normal', 'Benign', 'Malignant']
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (Validation Set)')
plt.show()

## 7. Generate Submission File

In [ ]:
@torch.no_grad()
def generate_predictions(model, loader, device):
    """Generate predictions for test set."""
    model.eval()
    predictions = []
    graph_ids = []
    
    for batch in tqdm(loader, desc='Generating predictions'):
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)
        preds = out.argmax(dim=-1).cpu().numpy()
        
        predictions.extend(preds)
        graph_ids.extend(batch.graph_id)
    
    return graph_ids, predictions

# Generate predictions
graph_ids, test_preds = generate_predictions(model, test_loader, device)

# Create submission dataframe
submission = pd.DataFrame({
    'graph_id': graph_ids,
    'prediction': test_preds
})

# Save submission
submission.to_csv('submission.csv', index=False)
print(f"\n✅ Submission saved to submission.csv")
print(f"   Rows: {len(submission)}")
print(f"\nPrediction distribution:")
print(submission['prediction'].value_counts().sort_index())

In [ ]:
# Validate submission
!python ../submission/validate.py --submission submission.csv

## 8. Next Steps

To improve your submission, consider:

1. **Try different architectures**: GAT, GraphSAGE, or custom models
2. **Hyperparameter tuning**: Hidden size, layers, learning rate
3. **Data augmentation**: Node dropout, edge perturbation
4. **Ensemble methods**: Combine multiple models
5. **Advanced pooling**: Attention-based or hierarchical pooling

Good luck! 🔬